# t_weight 快照与演化
**回答的问题：t 自适应学到了什么样的采样分布。**

t_weight / d_table 都存在 checkpoint buffer 里，纯读取、零前向。
注意：重构前（顶层键名）的旧 ckpt 读出来是初始均匀值。

In [ ]:
import os, sys
import torch

# ---- 路径：notebook 在 visual/ 下，切到 mmsegmentation 作为工作目录 ----
MMSEG = os.path.abspath('../tasks/Segmentation/mmsegmentation')
os.chdir(MMSEG)
sys_path_added = []
for p in [MMSEG, os.path.join(MMSEG, 'eden')]:
    import sys
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin, eden.pcgrad_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)
sys.modules.setdefault('mydiffae.pcgrad_mixin', eden.pcgrad_mixin)

# 注册 adapter（FCNWithDiffAEPlusPCGrad）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

sys.path.insert(0, os.path.abspath('../visual'))
import probe_core as pc


In [ ]:
CFG = 'configs/fcn/fcn_r50-d8_4xb4-80k_ade20k-512x512_diffae_tweight.py'
CKPTS = [
    'work_dirs/eden/fcn_diffae_tweight/iter_20000.pth',
    'work_dirs/eden/fcn_diffae_tweight/iter_40000.pth',
    'work_dirs/eden/fcn_diffae_tweight/iter_80000.pth',
]
OUT = 'work_dirs/offline_probe/tweight'
os.makedirs(OUT, exist_ok=True)

In [ ]:
snaps, labels = [], []
for ckpt in CKPTS:
    model, _ = pc.build_model(CFG, ckpt)
    snap = pc.read_tweight_snapshot(model)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    snaps.append(snap); labels.append(tag)
    pc.plot_tweight(snap, title=tag,
                    save_path=os.path.join(OUT, f'{tag}_tweight.png'))
    del model; torch.cuda.empty_cache()
print("done")

In [ ]:
fig = pc.plot_tweight_evolution(
    snaps, labels, save_path=os.path.join(OUT, "evolution.png"))
fig